In [1]:
# Mount Google Drive to access dataset files stored in Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import requests
import time

df = pd.read_csv("/content/drive/My Drive/EADA/EADA_DeepLearning/Project/merged_filled.csv")

/tmp/ipykernel_21095/4182090412.py:5: DtypeWarning: Columns (0,4,11,12,13,17,18,19,26,27,31) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/drive/My Drive/EADA/EADA_DeepLearning/Project/merged_filled.csv")


In [3]:
# =========================
# GOOGLE BOOKS FETCH
# =========================

def fetch_pub_data(isbn, title):
    try:
        # Priority 1: ISBN
        if pd.notna(isbn) and str(isbn) != "":
            url = f"https://www.googleapis.com/books/v1/volumes?q=isbn:{isbn}"
        else:
            url = f"https://www.googleapis.com/books/v1/volumes?q=intitle:{title}"

        r = requests.get(url).json()

        if "items" not in r:
            return None, None

        info = r["items"][0]["volumeInfo"]

        raw_date = info.get("publishedDate", None)

        if raw_date:
            year = raw_date[:4]  # safe extraction
        else:
            year = None

        return raw_date, year

    except:
        return None, None

In [4]:
# =========================
# APPLY TO ALL ROWS
# =========================

full_dates = []
years = []

for i, row in df.iterrows():
    isbn = row.get("isbn_13", None)
    title = row.get("title", "")

    full, year = fetch_pub_data(isbn, title)

    full_dates.append(full)
    years.append(year)

    if i % 100 == 0:
        print(f"Processed {i}")

    time.sleep(0.1)  # prevent rate limiting



Processed 0
Processed 100
Processed 200
Processed 300
Processed 400
Processed 500
Processed 600
Processed 700
Processed 800
Processed 900
Processed 1000
Processed 1100
Processed 1200
Processed 1300
Processed 1400
Processed 1500
Processed 1600
Processed 1700
Processed 1800
Processed 1900
Processed 2000
Processed 2100
Processed 2200
Processed 2300
Processed 2400
Processed 2500
Processed 2600
Processed 2700
Processed 2800
Processed 2900
Processed 3000
Processed 3100
Processed 3200
Processed 3300
Processed 3400
Processed 3500
Processed 3600
Processed 3700
Processed 3800
Processed 3900
Processed 4000
Processed 4100
Processed 4200
Processed 4300
Processed 4400
Processed 4500
Processed 4600
Processed 4700
Processed 4800
Processed 4900
Processed 5000
Processed 5100
Processed 5200
Processed 5300
Processed 5400
Processed 5500
Processed 5600
Processed 5700
Processed 5800
Processed 5900
Processed 6000
Processed 6100
Processed 6200
Processed 6300
Processed 6400
Processed 6500
Processed 6600
Process

In [5]:
# =========================
# ASSIGN NEW COLUMNS
# =========================

df["published_date_full"] = full_dates
df["published_year"] = years

In [6]:
# =========================
# RECOVER PUBLISHED YEAR (ROWS 12499+)
# SOURCE: books (2).csv
# =========================

# Load original books dataset (if not yet loaded)
books_df = pd.read_csv("/content/drive/My Drive/EADA/EADA_DeepLearning/Project/books.csv")

# Normalize for matching (same logic as before)
def norm(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

books_df["isbn13"] = books_df["isbn13"].apply(norm)
books_df["isbn10"] = books_df["isbn10"].apply(norm)
books_df["title"] = books_df["title"].apply(norm)
books_df["subtitle"] = books_df["subtitle"].apply(norm)

df["isbn_13"] = df["isbn_13"].apply(norm)
df["isbn_10"] = df["isbn_10"].apply(norm)
df["title"] = df["title"].apply(norm)
df["subtitle"] = df["subtitle"].apply(norm)

# Create matching key (same structure used earlier)
books_df["key"] = (
    books_df["isbn13"] + "|" +
    books_df["isbn10"] + "|" +
    books_df["title"] + "|" +
    books_df["subtitle"]
)

df["key"] = (
    df["isbn_13"] + "|" +
    df["isbn_10"] + "|" +
    df["title"] + "|" +
    df["subtitle"]
)

# Build lookup dictionary (key → published_year)
year_lookup = dict(zip(books_df["key"], books_df["published_year"]))


# =========================
# APPLY ONLY TO ROWS >= 12499
# =========================

START_IDX = 12499

df.loc[START_IDX:, "published_year"] = df.loc[START_IDX:, "key"].map(year_lookup)

# Derive published_date_full from year (fallback format)
df.loc[START_IDX:, "published_date_full"] = df.loc[START_IDX:, "published_year"].apply(
    lambda y: str(int(y)) if pd.notna(y) else None
)

In [7]:
# =========================
# SAVE FINAL DATASET
# =========================

output_path = "/content/drive/My Drive/EADA/EADA_DeepLearning/Project/book_recommender_dataset.csv"

df.to_csv(output_path, index=False)

print(f"File saved successfully: {output_path}")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

File saved successfully: /content/drive/My Drive/EADA/EADA_DeepLearning/Project/book_recommender_dataset.csv
Total rows: 19054
Total columns: 34
